# CryptoHypothesisLab — 06: Pre-Move V3 (Development, offline, matched controls)

**Samo u POTPUNO NOVOJ Colab sesiji, bez montiranja Google Drivea.** Koristi *isti* prethodno napravljen `HYPOLAB_MICRO_DEV_YYYY-MM_*.zip` sa svog računara. **Ne treba ponavljati 04 niti ponovo čitati izvorni C1/C2.**

- Svaka zatvorena 5s odluka na fiksnoj mreži ima jednake uslove. Prošlost mora biti mirna (strogo max-min u prethodnih 60s), kontrolni periodi se biraju bez budućih ishoda.
- 7 pravila, 15/30bps, 60/300s; sve 28 kombinacija, uključujući negativne i nedovoljne uzorke.
- Plan se čuva i preuzima **prije** računanja budućih ishoda. Rezultati, ZIP i plan su privatni; ništa ne objavljuj na GitHubu.
- Nema OOS, dokazanog edgea, bid/ask cijena, fillova ili stvarnog L2/C3.


In [ ]:
# 1. Sigurnosni preflight: izričito NEMA Drive mounta ni tokena.
import os,subprocess,sys,hashlib,json
from pathlib import Path
if any(Path(p).exists() for p in ('/content/drive/MyDrive','/content/gdrive/MyDrive','/gdrive/MyDrive')):
    raise RuntimeError('Drive je montiran: obrisi runtime i otvori novu sesiju.')
if any(os.environ.get(k) for k in ('GOOGLE_APPLICATION_CREDENTIALS','GOOGLE_OAUTH_ACCESS_TOKEN','GOOGLE_DRIVE_TOKEN')):
    raise RuntimeError('Google token/credential: zabranjen analiticki runtime')
repo=Path('/content/hypolab-v3-research')
if repo.exists():raise RuntimeError('Potreban je novi, cisti Colab runtime')
subprocess.run(['git','clone','--depth','1','https://github.com/DuStER1921/CryptoHypothesisLab.git',str(repo)],check=True,stdout=subprocess.DEVNULL)
commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip()
sys.path.insert(0,str(repo/'src'))
from hypolab.micro_bundle import inspect_micro_bundle,MAX_ARCHIVE_BYTES
from hypolab.premove_v3 import make_v3_plan,run_v3
from google.colab import files
print('OFFLINE PREFLIGHT PASS; code commit:',commit)


In [ ]:
# 2. Privatni vec pripremljeni MICRO ZIP + nezavisni SHA-256.
uploaded=files.upload()
if len(uploaded)!=1:raise RuntimeError('Dozvoljen je tacno jedan V2 MICRO ZIP')
name,payload=next(iter(uploaded.items()))
if not name.startswith('HYPOLAB_MICRO_DEV_') or not name.endswith('.zip') or not 0<len(payload)<=MAX_ARCHIVE_BYTES:
    raise RuntimeError('Pogresna vrsta ili velicina paketa')
expected=input('MICRO ZIP SHA-256 iz 04 ili Get-FileHash: ').strip().lower()
actual=hashlib.sha256(payload).hexdigest()
if actual!=expected:raise RuntimeError('MICRO SHA-256 mismatch')
private=Path('/content/hypolab-v3-private');private.mkdir(mode=0o700,exist_ok=False)
packet=private/'micro.zip'
with packet.open('xb') as f:
    os.chmod(packet,0o600)
    f.write(payload)
del uploaded,payload
metadata,c1,_=inspect_micro_bundle(packet,collect_5s=False)
print('MICRO V3 INPUT PASS:',metadata['month'],metadata['counts'])
print('C3 bookDepth: UNKNOWN_NOT_INCLUDED')


In [ ]:
# 3. Fiksni V3 plan: preuzmi GA PRIJE racunanja bilo kojeg buduceg ishoda.
plan=make_v3_plan(packet)
plan_path=private/'v3-registered-plan.json'
with plan_path.open('xb') as f:f.write(plan)
print('Plan SHA-256:',hashlib.sha256(plan).hexdigest())
files.download(str(plan_path))


In [ ]:
# 4. Sve predeklarisane hipoteze, ukljucujuci kontrolne uzorke i promašaje.
report=run_v3(packet,plan)
print('V3 REPORT:',report['schema'],'| odluke:',report['eligible_risk_decisions'])
print('Retrospektivne reference:',report['retrospective_references']['reference_count'])
print('NAPOMENA: isti vec analizirani Development mjesec; nijedna prednost NIJE nezavisno potvrdjena.')
print('Pravilo | cilj/horizont | TRAIN alert% / kontrola% / razlika pp | kasniji DEV alert% / kontrola% | matched N')
for t in report['trials']:
    a=t['train'];b=t['later_development_diagnostic']
    def pct(x):return 'n/a' if x is None else f'{100*x:.2f}%'
    def pp(x):return 'n/a' if x is None else f'{100*x:+.2f}pp'
    print(f"{t['rule']:29s} {t['threshold_bps']:>2}bps/{t['horizon_seconds']:>3}s | "
          f"{pct(a['alert_event_rate'])} / {pct(a['control_event_rate'])} / {pp(a['event_risk_difference'])} | "
          f"{pct(b['alert_event_rate'])} / {pct(b['control_event_rate'])} | "
          f"train={a['pairs']} later={b['pairs']}")


In [ ]:
# 5. Grafikon: alarm vs slicna kontrola. Samo DESKRIPTIVNO, bez profit claim.
import matplotlib.pyplot as plt
selected=[t for t in report['trials'] if t['threshold_bps']==15 and t['horizon_seconds']==300]
labels=[t['rule'] for t in selected]
a=[100*t['train']['alert_event_rate'] if t['train']['alert_event_rate'] is not None else 0 for t in selected]
b=[100*t['train']['control_event_rate'] if t['train']['control_event_rate'] is not None else 0 for t in selected]
fig,ax=plt.subplots(figsize=(12,5))
xs=range(len(selected))
ax.bar([x-.2 for x in xs],a,width=.4,label='Signal: Development train')
ax.bar([x+.2 for x in xs],b,width=.4,label='Bez signala: match kontrola')
ax.set_xticks(list(xs),labels,rotation=30,ha='right')
ax.set_ylabel('Future move >= 15 bps u 300s (%)')
ax.set_title('V3: ista rizicna mreza; prazne/nedovoljne grupe nisu dokaz')
ax.legend();ax.grid(True,axis='y',alpha=.2);plt.tight_layout();plt.show()
# Retrospektivni primjer: sam po sebi NIJE dokaz ranog predvidjanja.
if report['retrospective_gallery']:
    case=report['retrospective_gallery'][0];rows=case['timeline']
    xs=[r['offset_seconds'] for r in rows]
    fig,axes=plt.subplots(4,1,figsize=(12,9),sharex=True)
    series=[('last_trade_price','C2 trade-close'),('delta_btc','C2 delta BTC'),
            ('executed_quote_notional','Izvrseni notional po 5s'),('aggregate_rows','C2 agregirane trgovine')]
    for ax,(key,desc) in zip(axes,series):
        ax.plot(xs,[r[key] for r in rows],marker='.',linewidth=1)
        ax.axvline(0,ls='--',color='gray');ax.set_ylabel(desc);ax.grid(True,alpha=.2)
    axes[-1].set_xlabel('Sekunde od nastanka signala (ex-post prikaz)')
    fig.suptitle('PRVI hronoloski '+case['class']+' signal; mehanizam '+case['rule'])
    plt.tight_layout();plt.show()


In [ ]:
# 6. Preuzmi kompletan PRIVATNI V3 report (svi triali, negativni i unmatched).
report_path=private/'premove-v3-matched-report.json'
with report_path.open('x',encoding='utf-8') as f:
    json.dump(report,f,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)
    f.write('\n')
files.download(str(report_path))
print('Sacuvaj V3 plan i izvjestaj privatno. Nikada ne objavljuj na javni GitHub.')
